In [23]:
import matplotlib.font_manager as fm
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import logging

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

fontpath = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
fontprop = fm.FontProperties(fname=fontpath, size=12)
plt.rcParams["font.family"] = fontprop.get_name()

print(f"설정된 폰트: {fontprop.get_name()}")

설정된 폰트: NanumBarunGothic


In [24]:
import os
import re
import urllib.request
import zipfile
import sentencepiece as spm
import pandas as pd

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)

2.7.1+cu126


In [25]:
dataset_dir = os.path.join("./data")
ko_file = os.path.join(dataset_dir, "korean-english-park.train.ko")
en_file = os.path.join(dataset_dir, "korean-english-park.train.en")

# 각 파일을 읽어서 리스트로 저장
with open(ko_file, 'r', encoding='utf-8') as f:
    korean_sentences = [line.strip() for line in f.readlines()]

with open(en_file, 'r', encoding='utf-8') as f:
    english_sentences = [line.strip() for line in f.readlines()]

# 데이터프레임 생성
df = pd.DataFrame({
    'kor': korean_sentences,
    'eng': english_sentences
})

df

,kor,eng
0,"개인용 컴퓨터 사용의 상당 부분은 ""이것보다 뛰어날 수 있느냐?""","Much of personal computing is about ""can you t..."
1,모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하...,so a mention a few weeks ago about a rechargea...
2,그러나 이것은 또한 책상도 필요로 하지 않는다.,"Like all optical mice, But it also doesn't nee..."
3,"79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분...",uses gyroscopic sensors to control the cursor ...
4,정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔...,Intelligence officials have revealed a spate o...
...,...,...
94118,“우리는 3월 8일 김승연 회장과 그의 아들이 보복폭행에 가담한 혐의를 찾기 위해 ...,””We are hoping to seize material evidence to ...
94119,월요일 술집 종업원 6명은 김회장과 아들에게 폭행을 당했음을 진술했다고 경찰은 말했다.,"” On Monday, police secured statements from si..."
94120,그러나 불충분한 증거 확보로 수사에 어려움이 있다.,But the lack of material evidence is making it...
94121,김회장과 그의 아들은 보복폭행 혐의를 강력히 부인하고 있다.,Kim and his son both deny the allegations.


- 전처리

In [26]:
def preprocess_sentence(sentence):
    sentence = sentence.strip()
    # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    # 여러 공백을 하나로 통합
    sentence = re.sub(r'[" "]+', " ", sentence)
    # 한글, 영어, 숫자, 구두점만 남기고 나머지 제거
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,\s]+", " ", sentence)
    sentence = sentence.strip()
    return sentence


df['kor'] = df['kor'].apply(preprocess_sentence)
df['eng'] = df['eng'].apply(preprocess_sentence)

# 테스트
test_sentence = "개인용 컴퓨터 사용의 상당 부분은 '이것보다 뛰어날 수 있느냐?'"
print("원본:", test_sentence)
print("전처리 후:", preprocess_sentence(test_sentence))

원본: 개인용 컴퓨터 사용의 상당 부분은 '이것보다 뛰어날 수 있느냐?'
전처리 후: 개인용 컴퓨터 사용의 상당 부분은  이것보다 뛰어날 수 있느냐 ?


- 중복 제거

In [27]:
df = df.drop_duplicates().reset_index(drop=True)
df

,kor,eng
0,개인용 컴퓨터 사용의 상당 부분은 이것보다 뛰어날 수 있느냐 ?,Much of personal computing is about can you to...
1,모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하...,so a mention a few weeks ago about a rechargea...
2,그러나 이것은 또한 책상도 필요로 하지 않는다 .,"Like all optical mice , But it also doesn t ne..."
3,"79 . 95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목 , 팔 , 그외에 어...",uses gyroscopic sensors to control the cursor ...
4,정보 관리들은 동남 아시아에서의 선박들에 대한 많은 테러 계획들이 실패로 돌아갔...,Intelligence officials have revealed a spate o...
...,...,...
78931,우리는 3월 8일 김승연 회장과 그의 아들이 보복폭행에 가담한 혐의를 찾기 위해 총...,We are hoping to seize material evidence to pr...
78932,월요일 술집 종업원 6명은 김회장과 아들에게 폭행을 당했음을 진술했다고 경찰은 말했다 .,"On Monday , police secured statements from six..."
78933,그러나 불충분한 증거 확보로 수사에 어려움이 있다 .,But the lack of material evidence is making it...
78934,김회장과 그의 아들은 보복폭행 혐의를 강력히 부인하고 있다 .,Kim and his son both deny the allegations .


In [28]:
from konlpy.tag import Mecab

mecab = Mecab(dicpath="/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic")

def tokenize_korean(sentence):
    """한국어 문장을 Mecab으로 토큰화"""
    return mecab.morphs(sentence)

def tokenize_english(sentence):
    """영어 문장에 시작/끝 토큰 추가 후 토큰화"""
    sentence = "<start> " + sentence + " <end>"
    return sentence.split()


df['kor_tokens'] = df['kor'].apply(tokenize_korean)
df['eng_tokens'] = df['eng'].apply(tokenize_english)
print("토큰화 완료")

토큰화 완료


In [22]:
print("\n토큰화 결과 샘플:")
for i in range(2):
    print(f"원본 한국어: {df['kor'].iloc[i]}")
    print(f"토큰화된 한국어: {df['kor_tokens'].iloc[i]}")
    print(f"원본 영어: {df['eng'].iloc[i]}")
    print(f"토큰화된 영어: {df['eng_tokens'].iloc[i]}")
    print("-" * 50, '\n')

# 토큰 개수 통계
print(f"\n평균 한국어 토큰 수: {df['kor_tokens'].apply(len).mean():.2f}")
print(f"평균 영어 토큰 수: {df['eng_tokens'].apply(len).mean():.2f}")


토큰화 결과 샘플:
원본 한국어: 개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
토큰화된 한국어: ['개인', '용', '컴퓨터', '사용', '의', '상당', '부분', '은', '"', '이것', '보다', '뛰어날', '수', '있', '느냐', '?', '"']
원본 영어: Much of personal computing is about "can you top this?"
토큰화된 영어: ['<start>', 'Much', 'of', 'personal', 'computing', 'is', 'about', '"can', 'you', 'top', 'this?"', '<end>']
-------------------------------------------------- 

원본 한국어: 모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하지 않는다.
토큰화된 한국어: ['모든', '광', '마우스', '와', '마찬가지', '로', '이', '광', '마우스', '도', '책상', '위', '에', '놓', '는', '마우스', '패드', '를', '필요', '로', '하', '지', '않', '는다', '.']
원본 영어: so a mention a few weeks ago about a rechargeable wireless optical mouse brought in another rechargeable, wireless mouse.
토큰화된 영어: ['<start>', 'so', 'a', 'mention', 'a', 'few', 'weeks', 'ago', 'about', 'a', 'rechargeable', 'wireless', 'optical', 'mouse', 'brought', 'in', 'another', 'rechargeable,', 'wireless', 'mouse.', '<end>']
-------------------------------------------------- 


평

In [35]:
import itertools
from collections import Counter

class Vocab:
    def __init__(self, tokens_list, min_freq=2, max_size=None,
                 specials=("<pad>", "<bos>", "<eos>", "<unk>")):
        self.specials = list(specials)
        counter = Counter(itertools.chain.from_iterable(tokens_list))

        # specials 먼저
        self.itos = list(self.specials)

        # 빈도 내림차순(+토큰 이름 오름차순 tie-break)으로 정렬
        items = [(tok, freq) for tok, freq in counter.items()
                 if tok not in self.specials and freq >= min_freq]
        items.sort(key=lambda x: (-x[1], x[0]))

        if max_size is not None:
            remain = max(0, max_size - len(self.itos))
            items = items[:remain]

        self.itos += [tok for tok, _ in items]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self): return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(t, self.unk_id) for t in tokens]

    def decode(self, ids, skip_specials=True):
        out = []
        for i in ids:
            tok = self.itos[int(i)]
            if skip_specials and tok in self.specials:
                continue
            out.append(tok)
        return out

# 3) 영문 토큰은 이미 "<start> ... <end>" 포함 → (decoder 입력/정답) 분리 시에만 BOS/EOS로 매핑
# 단어장 생성 시에는 전체 토큰 분포를 반영하기 위해 원형을 사용
kor_vocab = Vocab(df['kor_tokens'].tolist(), min_freq=2, max_size=40)
eng_vocab = Vocab(df['eng_tokens'].tolist(), min_freq=2, max_size=40)

print(f"kor_vocab size: {len(kor_vocab)}, eng_vocab size: {len(eng_vocab)}")
print("BOS/EOS/PAD/UNK ids (eng):", eng_vocab.bos_id, eng_vocab.eos_id, eng_vocab.pad_id, eng_vocab.unk_id)

kor_vocab size: 11516, eng_vocab size: 11941
BOS/EOS/PAD/UNK ids (eng): 1 2 0 3


In [36]:
class NMTDataset(Dataset):
    def __init__(self, df, kor_vocab, eng_vocab):
        self.df = df
        self.kv = kor_vocab
        self.ev = eng_vocab

    def _eng_io(self, tokens):
        # tokens: ["<start>", w1, ..., wN, "<end>"]
        trg_in, trg_out = [], []

        for t in tokens:
            if t == "<start>":
                trg_in.append("<bos>")  # 입력은 BOS
                # 출력에는 포함하지 않음
            elif t == "<end>":
                trg_out.append("<eos>")  # 출력은 EOS로 마무리
            else:
                trg_in.append(t)
                trg_out.append(t)
        return trg_in, trg_out

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        kor_tokens = self.df.iloc[idx]['kor_tokens']   # encoder 입력
        eng_tokens = self.df.iloc[idx]['eng_tokens']   # decoder 원본("<start>", "<end>" 포함)

        trg_in_tokens, trg_out_tokens = self._eng_io(eng_tokens)

        src_ids = torch.tensor(self.kv.encode(kor_tokens), dtype=torch.long)
        trg_in_ids = torch.tensor(self.ev.encode(trg_in_tokens), dtype=torch.long)
        trg_out_ids = torch.tensor(self.ev.encode(trg_out_tokens), dtype=torch.long)
        return src_ids, trg_in_ids, trg_out_ids

def collate_fn(batch):
    # batch: list of (src_ids, trg_in_ids, trg_out_ids), 각 길이 가변
    src_seqs, trg_in_seqs, trg_out_seqs = zip(*batch)
    # pad_sequence: (max_len, batch)
    src_pad = pad_sequence(src_seqs, batch_first=False, padding_value=kor_vocab.pad_id)
    trg_in_pad = pad_sequence(trg_in_seqs, batch_first=False, padding_value=eng_vocab.pad_id)
    trg_out_pad = pad_sequence(trg_out_seqs, batch_first=False, padding_value=eng_vocab.pad_id)
    return src_pad.to(device), trg_in_pad.to(device), trg_out_pad.to(device)


train_df, valid_df = train_test_split(df, test_size=0.05, random_state=42, shuffle=True)
train_ds = NMTDataset(train_df.reset_index(drop=True), kor_vocab, eng_vocab)
valid_ds = NMTDataset(valid_df.reset_index(drop=True), kor_vocab, eng_vocab)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=False, collate_fn=collate_fn)

len(train_ds), len(valid_ds)

(74989, 3947)

In [37]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch_size, hidden_dim)
        # encoder_outputs: (src_len, batch_size, hidden_dim)

        src_len = encoder_outputs.shape[0]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch_size, src_len, hidden_dim)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # (batch_size, src_len, hidden_dim)

        energy = torch.tanh(self.W1(encoder_outputs) + self.W2(hidden))  # (batch_size, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)  # (batch_size, src_len)

        return nn.functional.softmax(attention, dim=1)  # (batch_size, src_len)


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        # src : (src_len, batch_size)
        embedded = self.embedding(src)  # embedded : (src_len, batch_size, emb_dim)
        outputs, hidden = self.rnn(embedded)  # outputs : (src_len, batch_size, hidden_dim)

        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention):
        super(Decoder, self).__init__()

        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        # Decoder RNN에는 embedding만 입력
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        # 출력층에는 hidden state와 attention value가 결합되어 입력
        self.fc_out = nn.Linear(hidden_dim + hidden_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        # input : (batch_size,)
        # hidden : (batch_size, hidden_dim)
        # encoder_outputs : (src_len, batch_size, hidden_dim)
        input = input.unsqueeze(0)  # input : (1, batch_size)
        embedded = self.embedding(input)  # embedded : (1, batch_size, emb_dim)

        # attention distribution을 계산합니다. decoder의 이전 hidden state, s_{t-1}와 encoder의 H가 입력됩니다.
        a = self.attention(hidden[-1], encoder_outputs)  # a : (batch_size, src_len)

        # H에 가중치를 부여해 attention value(Context vector) 계산
        a = a.unsqueeze(1)  # a : (batch_size, 1, src_len)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)  # encoder_outputs : (batch_size, src_len, hidden_dim)
        context = torch.bmm(a, encoder_outputs)  # context : (batch_size, 1, hidden_dim)
        context = context.permute(1, 0, 2)  # context : (1, batch_size, hidden_dim)

        output, hidden = self.rnn(embedded, hidden)

        # 출력층에서는 현재 hidden state와 context vector를 결합하여 예측값 생성
        output = output.squeeze(0)  # output : (batch_size, hidden_dim)
        context = context.squeeze(0)  # context : (batch_size, hidden_dim)
        prediction = self.fc_out(torch.cat((output, context), dim=1))  # (batch_size, output_dim)

        return prediction, hidden, a.squeeze(1)


class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=30, bos_id=1, eos_id=2):
        # 학습 모드에서는 trg_len 사용, 추론 모드에서는 max_len까지 동적 생성
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        # 조기 종료를 위해 tensor가 아닌 리스트 사용
        outputs = []

        # 시각화를 위해 attention 저장
        attentions = []

        # 인코더를 통해 context 생성
        encoder_outputs, hidden = self.encoder(src)

        if trg is not None:
            for t in range(0, trg.shape[0]):
                input = trg[t]
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))

        else:
            # inference에서는 target(정답)이 없기 때문에 sos_token을 생성해줍니다.
            input = torch.full((batch_size,), bos_id, dtype=torch.long, device=self.device)
            finished = torch.zeros(batch_size, dtype=torch.bool, device=self.device)

            for t in range(max_len):
                output, hidden, attention = self.decoder(input, hidden, encoder_outputs)
                outputs.append(output.unsqueeze(0))
                attentions.append(attention.unsqueeze(0))
                top1 = output.argmax(1)
                input = top1

                # 조기 종료 조건
                finished |= (top1 == eos_id)
                if finished.all():
                    break

        outputs = torch.cat(outputs, dim=0)  # (trg_len, batch_size, output_dim)
        attentions = torch.cat(attentions, dim=0)  # (trg_len, batch_size, src_len)

        return outputs, attentions

In [38]:
# 하이퍼파라미터
INPUT_DIM = len(kor_vocab)
OUTPUT_DIM = len(eng_vocab)
EMB_DIM = 256
HID_DIM = 512
LR = 2e-3
EPOCHS = 10
CLIP = 1.0

# 모델 구성
attn = BahdanauAttention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, attn)
model = Seq2SeqAttention(enc, dec, device).to(device)
print(model)

Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(11516, 256)
    (rnn): GRU(256, 512)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(11941, 256)
    (rnn): GRU(256, 512)
    (fc_out): Linear(in_features=1024, out_features=11941, bias=True)
  )
)


In [39]:
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=eng_vocab.pad_id)

In [40]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

def train_one_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    running = 0.0
    for src, trg_in, trg_out in tqdm(loader, desc="Train", leave=False):
        optimizer.zero_grad()
        # model forward: trg=decoder 입력, 출력 shape: (trg_len, batch, vocab)
        logits, _att = model(src, trg=trg_in, bos_id=eng_vocab.bos_id, eos_id=eng_vocab.eos_id)
        # CE 계산: (N, C) vs (N,)
        # logits: (T, B, V) -> (T*B, V); target: (T, B) -> (T*B,)
        loss = criterion(logits.view(-1, logits.size(-1)), trg_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        running += loss.item()
    return running / max(1, len(loader))

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total = 0.0
    for src, trg_in, trg_out in tqdm(loader, desc="Valid", leave=False):
        logits, _att = model(src, trg=trg_in, bos_id=eng_vocab.bos_id, eos_id=eng_vocab.eos_id)
        loss = criterion(logits.view(-1, logits.size(-1)), trg_out.reshape(-1))
        total += loss.item()
    return total / max(1, len(loader))

best_val = float("inf")
for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = evaluate(model, valid_loader, criterion)
    print(f"[Epoch {epoch:02d}] train loss: {tr_loss:.4f} | valid loss: {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "model": model.state_dict(),
            "kor_vocab": kor_vocab.__dict__,
            "eng_vocab": eng_vocab.__dict__,
            "config": dict(INPUT_DIM=INPUT_DIM, OUTPUT_DIM=OUTPUT_DIM, EMB_DIM=EMB_DIM, HID_DIM=HID_DIM)
        }, "best_seq2seq_attn.pt")
        print("  ↳ best model saved.")

[Epoch 01] train loss: 4.8591 | valid loss: 4.1678
  ↳ best model saved.


[Epoch 02] train loss: 3.6611 | valid loss: 3.9022
  ↳ best model saved.


[Epoch 03] train loss: 3.0200 | valid loss: 3.9292


[Epoch 04] train loss: 2.5700 | valid loss: 4.0437


[Epoch 05] train loss: 2.2549 | valid loss: 4.2033


[Epoch 06] train loss: 2.0300 | valid loss: 4.3548


[Epoch 07] train loss: 1.8618 | valid loss: 4.5245


[Epoch 08] train loss: 1.7282 | valid loss: 4.6751


[Epoch 09] train loss: 1.6264 | valid loss: 4.8312


[Epoch 10] train loss: 1.5469 | valid loss: 4.9760


In [41]:
@torch.no_grad()
def translate(sentence, max_len=50, show_attention=False):
    model.eval()
    # 1) 전처리 & 토크나이즈
    sent = preprocess_sentence(sentence)
    src_toks = tokenize_korean(sent)
    # 2) 인덱싱
    src_ids = torch.tensor(kor_vocab.encode(src_toks), dtype=torch.long, device=device)
    src = src_ids.unsqueeze(1)  # (src_len, 1)
    # 3) 디코딩
    logits, attn = model(src, trg=None, max_len=max_len,
                         bos_id=eng_vocab.bos_id, eos_id=eng_vocab.eos_id)
    # logits: (T, 1, V)
    pred_ids = logits.argmax(-1).squeeze(1)  # (T,)
    # EOS까지 자르기
    pred_list = []
    for idx in pred_ids.tolist():
        if idx == eng_vocab.eos_id:
            break
        if idx == eng_vocab.bos_id or idx == eng_vocab.pad_id:
            continue
        pred_list.append(idx)
    pred_tokens = eng_vocab.decode(pred_list, skip_specials=True)
    pred_text = " ".join(pred_tokens)

    if show_attention:
        # 주의: attn shape: (trg_len, batch=1, src_len)
        import matplotlib.pyplot as plt
        import numpy as np
        att = attn.squeeze(1).cpu().numpy()  # (T, S)
        fig, ax = plt.subplots(figsize=(min(12, len(src_toks)*0.6+2), 6))
        im = ax.imshow(att[:len(pred_tokens)], cmap="viridis", aspect='auto')
        ax.set_xticks(range(len(src_toks)))
        ax.set_xticklabels(src_toks, rotation=45, ha='right', fontsize=9)
        ax.set_yticks(range(len(pred_tokens)))
        ax.set_yticklabels(pred_tokens, fontsize=9)
        ax.set_xlabel("Source (Korean)")
        ax.set_ylabel("Predicted (English)")
        plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
        plt.title("Attention")
        plt.tight_layout()
        plt.show()

    return pred_text

# 샘플 n개 테스트
sample_idx = np.random.choice(len(valid_df), size=min(5, len(valid_df)), replace=False)
for i in sample_idx:
    src_kor = valid_df.iloc[i]['kor']
    tgt_eng = " ".join([t for t in valid_df.iloc[i]['eng_tokens'] if t not in ("<start>", "<end>")])
    pred = translate(src_kor, show_attention=False)
    print("-"*80)
    print("SRC:", src_kor)
    print("TGT:", tgt_eng)
    print("PRD:", pred)

--------------------------------------------------------------------------------
SRC: 건설 노동자인 24세의 파디 술레이만은 미국은 이스라엘을 지지해 우리나라를 계속 점령케 하고 있다 고 말했다 .
TGT: The United States is backing Israel to continue the occupation of our land , said 24 year old construction worker Fadi Suleiman .
PRD: The 24 year old is looking to Israel , a 24 year old , a 24th member of the group of in the East , told a U . S . official .
--------------------------------------------------------------------------------
SRC: 당국은 반다나가 과일과 단 음식을 요구해 음식을 로프에 매달아 내려 보냈다고 전했다 .
TGT: Officials said the girl frequently asked for fruits and sweets , which was sent to her with the help of a rope .
PRD: Officials say the late teens and spent time for sending food and veterinary cars nationwide had gone to the terms and food gap .
--------------------------------------------------------------------------------
SRC: 신지애는  올해는 내게 특별한 해 라고 밝혔다 .
TGT: A really special year for me , said Shin , who isn t even a full fledged member 

- Train Loss는 계속해서 안정적으로 감소하는데 validation loss는 초반에 감소했다가 이후 계속해서 증가
- 데이터가 너무 적어서 그런 것 같다.
- 정답 번역과 유사한 결과가 나온다고 보기에는 무리가 있지만, 키워드는 겹치게 잘 나오는 것 같다.